In [ ]:
import librosa
import numpy as np
from pathlib import Path
from typing import Union
from sklearn.model_selection import train_test_split

def extract_features(file_path: Union[str, Path], n_mfcc=13):
    """Extracts the average MFCC from an audio file."""

    y, sr = librosa.load(str(file_path), sr=16000)
    # Calculate MFCC
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc)
    # Average each feature to have one vector per file
    return np.mean(mfcc.T, axis=0)

def load_dataset(processed_path: Union[str, Path]):
    X, y = [], []
    processed_dir = Path(processed_path)
     # Mapping folder names to numbers (labels)
    label_map = {"tv": 0, "talk": 1, "yell": 2}
    
    for category_dir in processed_dir.iterdir():
        if not category_dir.is_dir():
            continue

        label = label_map.get(category_dir.name)
        if label is None:
            continue

        for file_path in category_dir.glob("*.wav"):
            features = extract_features(file_path=file_path)
            X.append(features)
            y.append(label)

    return np.array(X), np.array(y)

X, y = load_dataset('../data/processed')

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Number of samples in X: {len(X)}")
print(f"Training set size: {X_train.shape}")
print(f"Test set size: {X_test.shape}")
print(f"Unique labels in y: {np.unique(y)}")

Number of samples in X: 688
Training set size: (550, 13)
Test set size: (138, 13)
Unique labels in y: [0 1 2]
